# Лабораторная работа №1 — разведочный анализ данных

Разведочный анализ подготовленной Iceberg-таблицы средствами Apache Spark.

Основные этапы:

- анализ структуры;
- исследование пропусков;
- описательные статистики;
- категориальные распределения;
- квантили и IQR;
- поиск выбросов;
- распределения числовых признаков;
- корреляционный анализ.


## 1. Инициализация


In [ ]:
%matplotlib inline

from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.sql.functions import (
    col,
    count,
    desc,
    floor,
    isnan,
    max as spark_max,
    min as spark_min,
    sum as spark_sum,
    when,
)

from src.spark_session import create_spark

spark = create_spark("Lab1EDA")

TABLE_NAME = "local.lab1.used_cars"

RESULTS_DIR = Path("/app/output/results")
PLOTS_DIR = Path("/app/output/plots")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Загрузка очищенных данных


In [ ]:
df = spark.table(TABLE_NAME)

print("Количество строк:", f"{df.count():,}")
print("Количество столбцов:", len(df.columns))

df.printSchema()


## 3. Просмотр данных


In [ ]:
df.show(10, truncate=False)


## 4. Анализ пропущенных значений


In [ ]:
missing_expressions = []

for field in df.schema.fields:
    column = field.name

    if field.dataType.simpleString() in ("float", "double"):
        condition = (
            col(column).isNull()
            | isnan(col(column))
        )
    else:
        condition = col(column).isNull()

    missing_expressions.append(
        spark_sum(
            when(condition, 1).otherwise(0)
        ).alias(column)
    )

missing_row = (
    df.select(*missing_expressions)
    .collect()[0]
    .asDict()
)

row_count = df.count()

missing_data = []

for column, missing_count in missing_row.items():
    missing_count = int(missing_count or 0)

    missing_percent = (
        missing_count / row_count * 100
        if row_count
        else 0
    )

    missing_data.append(
        {
            "column": column,
            "missing_count": missing_count,
            "missing_percent": round(missing_percent, 2),
        }
    )

missing_df = (
    pd.DataFrame(missing_data)
    .sort_values(
        "missing_percent",
        ascending=False,
    )
)

missing_df


In [ ]:
missing_df.to_csv(
    RESULTS_DIR / "missing_values.csv",
    index=False,
)


## 5. Визуализация пропусков


In [ ]:
missing_plot = missing_df[
    missing_df["missing_percent"] > 0
]

plt.figure(figsize=(10, 6))

plt.barh(
    missing_plot["column"],
    missing_plot["missing_percent"],
)

plt.xlabel("Доля пропусков, %")
plt.ylabel("Признак")
plt.title("Пропущенные значения")
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "missing_values.png"
)

plt.show()


## 6. Числовые признаки


In [ ]:
NUMERIC_COLUMNS = [
    "daysonmarket",
    "horsepower",
    "maximum_seating",
    "mileage",
    "price",
    "year",
]

stats = df.select(
    *NUMERIC_COLUMNS
).describe()

stats.show(truncate=False)

stats_pd = stats.toPandas()
stats_pd


In [ ]:
stats_pd.to_csv(
    RESULTS_DIR / "numeric_statistics.csv",
    index=False,
)


## 7. Категориальные признаки


In [ ]:
body_type_distribution = (
    df.groupBy("body_type")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

body_type_distribution.show(
    30,
    truncate=False,
)


In [ ]:
body_pd = body_type_distribution.toPandas()

body_pd.to_csv(
    RESULTS_DIR / "body_type_distribution.csv",
    index=False,
)

plt.figure(figsize=(10, 6))

plt.barh(
    body_pd["body_type"].fillna("NULL"),
    body_pd["count"],
)

plt.xlabel("Количество")
plt.ylabel("Тип кузова")
plt.title("Распределение автомобилей по типу кузова")
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "body_type_distribution.png"
)

plt.show()


In [ ]:
wheel_distribution = (
    df.groupBy("wheel_system")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

wheel_distribution.show(
    30,
    truncate=False,
)


## 8. Квантили и поиск выбросов методом IQR


In [ ]:
outlier_rows = []

for column in NUMERIC_COLUMNS:
    clean = (
        df.select(column)
        .where(col(column).isNotNull())
    )

    q1, median, q3 = clean.approxQuantile(
        column,
        [0.25, 0.5, 0.75],
        0.01,
    )

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    lower_count = clean.where(
        col(column) < lower_bound
    ).count()

    upper_count = clean.where(
        col(column) > upper_bound
    ).count()

    valid_count = clean.count()

    outlier_rows.append(
        {
            "column": column,
            "q1": q1,
            "median": median,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "lower_outliers": lower_count,
            "upper_outliers": upper_count,
            "total_outliers": lower_count + upper_count,
            "valid_values": valid_count,
        }
    )

outliers_df = pd.DataFrame(outlier_rows)

outliers_df


In [ ]:
outliers_df.to_csv(
    RESULTS_DIR / "outliers_iqr.csv",
    index=False,
)


## 9. Boxplot по рассчитанным квартилям


In [ ]:
for _, row in outliers_df.iterrows():
    column = row["column"]

    stats = [{
        "label": column,
        "whislo": row["lower_bound"],
        "q1": row["q1"],
        "med": row["median"],
        "q3": row["q3"],
        "whishi": row["upper_bound"],
        "fliers": [],
    }]

    fig, ax = plt.subplots(figsize=(8, 3))

    ax.bxp(
        stats,
        vert=False,
    )

    ax.set_title(f"Boxplot: {column}")
    ax.set_xlabel(column)

    fig.tight_layout()

    fig.savefig(
        PLOTS_DIR / f"boxplot_{column}.png"
    )

    plt.show()
    plt.close(fig)


## 10. Распределения числовых признаков

Данные сначала агрегируются Spark по интервалам.  
В Pandas передаются только небольшие таблицы с количеством объектов в каждом интервале.


In [ ]:
def create_histogram(df, column, bins=30):
    bounds = (
        df.select(column)
        .where(col(column).isNotNull())
        .agg(
            spark_min(column).alias("min"),
            spark_max(column).alias("max"),
        )
        .collect()[0]
    )

    min_value = bounds["min"]
    max_value = bounds["max"]

    if min_value is None or max_value is None:
        return

    if min_value == max_value:
        return

    width = (max_value - min_value) / bins

    histogram = (
        df.select(column)
        .where(col(column).isNotNull())
        .withColumn(
            "bucket",
            floor(
                (col(column) - min_value) / width
            ),
        )
        .where(col("bucket") < bins)
        .groupBy("bucket")
        .agg(count("*").alias("count"))
        .orderBy("bucket")
    )

    pandas_hist = histogram.toPandas()

    pandas_hist["start"] = (
        min_value
        + pandas_hist["bucket"] * width
    )

    pandas_hist["end"] = (
        pandas_hist["start"] + width
    )

    pandas_hist.to_csv(
        RESULTS_DIR / f"histogram_{column}.csv",
        index=False,
    )

    plt.figure(figsize=(9, 5))

    plt.bar(
        pandas_hist["start"],
        pandas_hist["count"],
        width=width,
        align="edge",
    )

    plt.xlabel(column)
    plt.ylabel("Количество")
    plt.title(f"Распределение: {column}")
    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / f"histogram_{column}.png"
    )

    plt.show()
    plt.close()


for column in NUMERIC_COLUMNS:
    create_histogram(
        df,
        column,
    )


## 11. Корреляционный анализ


In [ ]:
correlation_matrix = pd.DataFrame(
    index=NUMERIC_COLUMNS,
    columns=NUMERIC_COLUMNS,
    dtype=float,
)

for column in NUMERIC_COLUMNS:
    correlation_matrix.loc[
        column,
        column,
    ] = 1.0

for first, second in combinations(
    NUMERIC_COLUMNS,
    2,
):
    pair = (
        df.select(first, second)
        .where(
            col(first).isNotNull()
            & col(second).isNotNull()
        )
    )

    correlation = pair.stat.corr(
        first,
        second,
    )

    correlation_matrix.loc[
        first,
        second,
    ] = correlation

    correlation_matrix.loc[
        second,
        first,
    ] = correlation

correlation_matrix


In [ ]:
correlation_matrix.to_csv(
    RESULTS_DIR / "correlation_matrix.csv"
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix.astype(float),
    annot=True,
    fmt=".2f",
    square=True,
)

plt.title("Корреляционная матрица")
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "correlation_matrix.png"
)

plt.show()


## 12. Итог

После выполнения анализа результаты сохраняются:

- таблицы — в `output/results`;
- графики — в `output/plots`.

При переходе на финальный датасет необходимо заменить набор признаков и предметно адаптировать этапы очистки и анализа.
